## Importando as Bibliotecas

In [1]:
# ================================
# Utilitários
# ================================
import time
import joblib
import duckdb

# ================================
# Manipulação de dados
# ================================
import pandas as pd
import numpy as np

# ================================
# Visualização de dados
# ================================
import matplotlib.pyplot as plt
import seaborn as sns

# ================================
# Divisão, validação e otimização
# ================================
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    cross_validate,
    StratifiedKFold,
    cross_val_score,
    learning_curve
)

# ================================
# Pré-processamento
# ================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

# ================================
# Modelos de Machine Learning
# ================================
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

import xgboost as xgb

# ================================
# Avaliação de modelos
# ================================
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

# ================================
# Interpretabilidade
# ================================
import shap

print("Bibliotecas importadas com sucesso!!")

Bibliotecas importadas com sucesso!!


## Leitura da Base de Dados

In [2]:
dados = pd.read_csv("../data/dados_housing.csv")
dados.head()

,longitude,latitude,idade_mediana_imoveis,total_comodos,total_quartos,populacao,domicilios,renda_media,valor_media_imovel,proximidade_oceano
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,proximo_baia
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,proximo_baia
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,proximo_baia
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,proximo_baia
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,proximo_baia


## Validação rápida dos Dados

In [3]:
print(f"Total de Linhas -> {dados.shape[0]}")
print(f"Total de Colunas -> {dados.shape[1]}")

Total de Linhas -> 20640
Total de Colunas -> 10


In [4]:
print(dados.info())

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   longitude              20640 non-null  float64
 1   latitude               20640 non-null  float64
 2   idade_mediana_imoveis  20640 non-null  float64
 3   total_comodos          20640 non-null  float64
 4   total_quartos          20433 non-null  float64
 5   populacao              20640 non-null  float64
 6   domicilios             20640 non-null  float64
 7   renda_media            20640 non-null  float64
 8   valor_media_imovel     20640 non-null  float64
 9   proximidade_oceano     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB
None


In [5]:
# Mostrando os Dados Nulos
print(dados.isna().sum())

longitude                  0
latitude                   0
idade_mediana_imoveis      0
total_comodos              0
total_quartos            207
populacao                  0
domicilios                 0
renda_media                0
valor_media_imovel         0
proximidade_oceano         0
dtype: int64


## Criando Variável para fazer Estratificação

In [6]:
dados["faixa_preco"] = pd.qcut(
    dados["valor_media_imovel"],
    q=5,
    labels=False
)

In [7]:
dados["faixa_preco"].value_counts()

faixa_preco
2    4132
1    4129
0    4129
4    4125
3    4125
Name: count, dtype: int64

## Divisão em X e Y - Separando Dados

In [8]:
X = dados.drop("valor_media_imovel", axis=1)
y = dados["valor_media_imovel"]
estrato = dados["faixa_preco"]

In [9]:
# 1º Divisão -> treino + validação e Teste
X_temp, X_teste, y_temp, y_teste, estrato_temp, estrato_teste = train_test_split(X, y, estrato, test_size=0.20, random_state=42, stratify=estrato)

# 2º Divisão -> treino e validação
X_treino, X_val, y_treino, y_val, estrato_treino, estrato_val = train_test_split(X_temp, y_temp, estrato_temp, test_size=0.25, random_state=42, stratify=estrato_temp)

In [10]:
print(f"Treino: {len(X_treino) / len(dados):.2%}")
print(f"Validação: {len(X_val) / len(dados):.2%}")
print(f"Teste: {len(X_teste) / len(dados):.2%}")

Treino: 60.00%
Validação: 20.00%
Teste: 20.00%


In [11]:
print(X_treino["faixa_preco"].value_counts())
print(X_val["faixa_preco"].value_counts())
print(X_teste["faixa_preco"].value_counts())

faixa_preco
2    2480
0    2477
1    2477
4    2475
3    2475
Name: count, dtype: int64
faixa_preco
2    826
0    826
1    826
4    825
3    825
Name: count, dtype: int64
faixa_preco
2    826
0    826
1    826
3    825
4    825
Name: count, dtype: int64


In [12]:
X_treino = X_treino.drop("faixa_preco", axis=1)
X_val = X_val.drop("faixa_preco", axis=1)
X_teste = X_teste.drop("faixa_preco", axis=1)

## Criando colunas do Feature Engineering

In [13]:
def criar_colunas(df):
    df = df.copy()
    df["comodos_por_domicilio"] = df["total_comodos"] / df["domicilios"]
    df["quartos_por_comodo"] = df["total_quartos"] / df["total_comodos"]

    return df

In [14]:
X_treino_fe = criar_colunas(X_treino)
X_teste_fe = criar_colunas(X_teste)
X_val_fe = criar_colunas(X_val)

## Pré-processamento dos Dados

In [15]:
col_numericas = ['idade_mediana_imoveis', 'total_comodos', 'total_quartos', 'populacao', 'domicilios', 'renda_media']
col_categoricas = ['proximidade_oceano']

col_numericas_fe = ['idade_mediana_imoveis', 'total_comodos', 'total_quartos', 'populacao', 'domicilios', 'renda_media',
                     'comodos_por_domicilio', 'quartos_por_comodo']

In [16]:
pipeline_numerico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [17]:
pipeline_categorico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", sparse_output=False))
    ]
)

In [18]:
preprocessador = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, col_numericas),
        ("cat", pipeline_categorico, col_categoricas)
    ],
    remainder="passthrough"
)

X_treino_tratado = preprocessador.fit_transform(X_treino)
X_val_tratado = preprocessador.transform(X_val)
X_teste_tratado = preprocessador.transform(X_teste)

In [19]:
preprocessador_fe = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, col_numericas_fe),
        ("cat", pipeline_categorico, col_categoricas)
    ],
    remainder="passthrough"
)

X_treino_fe_tratado = preprocessador_fe.fit_transform(X_treino_fe)
X_val_fe_tratado = preprocessador_fe.transform(X_val_fe)
X_teste_fe_tratado = preprocessador_fe.transform(X_teste_fe)

In [24]:
num_cols = list(col_numericas)
cat_cols = preprocessador.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(col_categoricas).tolist()

X_treino = pd.DataFrame(X_treino_tratado, columns=(num_cols) + (cat_cols) + ["latitude"] + ["longitude"])
X_teste = pd.DataFrame(X_teste_tratado, columns=(num_cols) + (cat_cols) + ["latitude"] + ["longitude"])
X_val = pd.DataFrame(X_val_tratado, columns=(num_cols) + (cat_cols) + ["latitude"] + ["longitude"])

In [26]:
num_cols_fe = list(col_numericas_fe)
cat_cols = preprocessador.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(col_categoricas).tolist()

X_treino_fe = pd.DataFrame(X_treino_fe_tratado, columns=(num_cols_fe) + (cat_cols) + ["latitude"] + ["longitude"])
X_teste_fe = pd.DataFrame(X_teste_fe_tratado, columns=(num_cols_fe) + (cat_cols) + ["latitude"] + ["longitude"])
X_val_fe = pd.DataFrame(X_val_fe_tratado, columns=(num_cols_fe) + (cat_cols) + ["latitude"] + ["longitude"])

In [25]:
X_treino.head()

,idade_mediana_imoveis,total_comodos,total_quartos,populacao,domicilios,renda_media,proximidade_oceano_interior,proximidade_oceano_menos_1h_oceano,proximidade_oceano_proximo_baia,proximidade_oceano_proximo_oceano,latitude,longitude
0,-0.843499,-1.052519,-1.099944,-1.108935,-1.133174,-1.399217,1.0,0.0,0.0,0.0,-120.54,37.68
1,1.870922,-0.663041,0.300097,-0.372001,0.328918,-1.279084,0.0,0.0,1.0,0.0,-122.40,37.79
2,-1.402350,0.569084,0.072350,0.227096,0.126434,1.194674,0.0,1.0,0.0,0.0,-116.80,32.80
3,0.034696,-0.137017,0.007622,-0.270199,0.110656,-0.778840,0.0,0.0,1.0,0.0,-122.09,37.68
4,-1.322514,2.044976,1.738494,-0.217046,0.189546,0.560171,1.0,0.0,0.0,0.0,-116.42,33.79


In [27]:
X_treino_fe.head()

,idade_mediana_imoveis,total_comodos,total_quartos,populacao,domicilios,renda_media,comodos_por_domicilio,quartos_por_comodo,proximidade_oceano_interior,proximidade_oceano_menos_1h_oceano,proximidade_oceano_proximo_baia,proximidade_oceano_proximo_oceano,latitude,longitude
0,-0.843499,-1.052519,-1.099944,-1.108935,-1.133174,-1.399217,-0.177517,0.236069,1.0,0.0,0.0,0.0,-120.54,37.68
1,1.870922,-0.663041,0.300097,-0.372001,0.328918,-1.279084,-1.459991,5.824336,0.0,0.0,1.0,0.0,-122.40,37.79
2,-1.402350,0.569084,0.072350,0.227096,0.126434,1.194674,0.689869,-1.135562,0.0,1.0,0.0,0.0,-116.80,32.80
3,0.034696,-0.137017,0.007622,-0.270199,0.110656,-0.778840,-0.458870,0.299362,0.0,0.0,1.0,0.0,-122.09,37.68
4,-1.322514,2.044976,1.738494,-0.217046,0.189546,0.560171,2.905542,-0.598131,1.0,0.0,0.0,0.0,-116.42,33.79
